# 🤖 Building an Agentic System

**Goal:** In this workshop, we will move beyond simple chatbots. We will build an **"Agentic System"**—a team of specialized AI workers managed by a central decision-maker.

---

## 1. Introduction: Why "Smart" Bots Act Stupid

### The "Black Box" Problem
Imagine you are building a customer support bot for a store. You ask it: *"What is the return policy for the iPhone 16?"*

A standard Large Language Model (LLM) like GPT-4 might say:
> *"You can return it within 14 days to any Apple Store."*

**The Problem?** This might be true for Apple *in general*, but it might be **false** for *your* specific store in Singapore. The AI is relying on its "frozen" training data from the internet, not your actual business rules. It is "hallucinating" a policy because it wants to be helpful.

---
### The Solution: Agentic RAG
We don't need a creative writer; we need a strict librarian. To fix this, we move from a simple chatbot to an **Agentic System**. Instead of one big brain trying to know everything, we build a **Team of Specialists**:

1.  **The Router (The Receptionist):** Identifies what the user wants (e.g., "This is a shipping question").
2.  **The Specialist (The Worker):** A specific agent (e.g., "Policy Agent") that *only* handles that topic.
3.  **The Knowledge Base (The Library):** The agent looks up the *exact* document before answering.

---
## 2. Setup & Dependencies

First, let's install the necessary tools. We are using **LangChain** (for logic), **LangGraph** (for the agent loop), and **ChromaDB** (for memory).

In [1]:
# 1️⃣ INSTALL DEPENDENCIES
print("⏳ Installing dependencies...")
!pip install -qU langchain-groq langchain-huggingface langchain-chroma langgraph langchain-community sentence-transformers unstructured chromadb gradio

import os
import chromadb
from google.colab import userdata, drive
from langchain_groq import ChatGroq

# Setup Google Drive
drive.mount('/content/drive')


⏳ Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.3/566.3 kB

---

## 3. Configuration

We define our environment variables here. This acts as the "control center" for our project.

In [19]:
# 2️⃣ CONFIGURATION
# Define where our data lives
SOURCE_DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/langchain/apple inc data for multi agent"
DRIVE_DB_PATH = "/content/drive/MyDrive/chroma_db"
EMBEDDING_MODEL = "all-mpnet-base-v2"

# Setup LLM (The Brain)
# We use OpenAI's open-source model (via Groq) for high-quality reasoning
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

print("✅ Environment Ready.")


✅ Environment Ready.


---

## 4. Provisioning the Knowledge Base

Before our agents can answer questions, they need a library of facts.

*Recall from our previous RAG lesson:* Computers don't read text; they read numbers (vectors). We must "chunk" our documents and embed them into a database.


### The Chunker Lab (Refresher)
Let's quickly verify how our AI "sees" text by testing the chunker on a dummy sentence.

In [3]:
# 🧪 CHUNKER LAB
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Try changing chunk_size to see how it affects the output
my_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
sample_text = "Artificial Intelligence has evolved from simple rule-based systems to complex agentic architectures. In a traditional RAG system, the process is linear: a user asks a question, the system retrieves documents, and the LLM generates an answer. However, this often lacks the reasoning depth required for enterprise tasks."

chunks = my_splitter.create_documents([sample_text])
print(f"📄 Split into {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"[Chunk {i}]: {chunk.page_content}")


📄 Split into 8 chunks:
[Chunk 0]: Artificial Intelligence has evolved from simple
[Chunk 1]: simple rule-based systems to complex agentic
[Chunk 2]: agentic architectures. In a traditional RAG
[Chunk 3]: RAG system, the process is linear: a user asks a
[Chunk 4]: asks a question, the system retrieves documents,
[Chunk 5]: and the LLM generates an answer. However, this
[Chunk 6]: this often lacks the reasoning depth required for
[Chunk 7]: for enterprise tasks.


### The Builder (Ingestion Engine)
Now, let's process your actual files from Google Drive.

In [4]:
# 🏗️ INGESTION ENGINE
from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize Client
persistent_client = chromadb.PersistentClient(path=DRIVE_DB_PATH)
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
real_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

def build_knowledge_base():
    if not os.path.exists(SOURCE_DATA_DIR):
        print(f"❌ Source directory '{SOURCE_DATA_DIR}' not found.")
        return

    print("\n🚀 Starting Ingestion...")
    # This deletes the old, duplicate collections so we start fresh.
    existing_collections = persistent_client.list_collections()
    for col in existing_collections:
        try:
            print(f"   🗑️  Cleaning up old collection: {col.name}")
            persistent_client.delete_collection(col.name)
        except:
            pass

    for item_name in os.listdir(SOURCE_DATA_DIR):
        if item_name.startswith('.'): continue
        file_path = os.path.join(SOURCE_DATA_DIR, item_name)
        safe_name = os.path.splitext(item_name)[0].replace(" ", "_").lower() + "_collection"

        try:
            if os.path.isdir(file_path):
                loader = DirectoryLoader(file_path, glob="**/*", loader_cls=UnstructuredFileLoader)
            else:
                loader = UnstructuredFileLoader(file_path)

            docs = loader.load()
            if not docs: continue

            splits = real_splitter.split_documents(docs)
            Chroma(client=persistent_client, collection_name=safe_name, embedding_function=embeddings).add_documents(splits)
            print(f"   ✅ Indexed: {item_name} -> {safe_name}")
        except Exception as e:
            print(f"   ❌ Error on {item_name}: {e}")

build_knowledge_base()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


🚀 Starting Ingestion...


/tmp/ipython-input-2167753160.py:35: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader(file_path)


   ❌ Error on dataset sources.docx: partition_docx() is not available because one or more dependencies are not installed. Use: pip install "unstructured[docx]" (including quotes) to install the required dependencies
   ✅ Indexed: policy -> policy_collection
   ✅ Indexed: product -> product_collection


---

## 5. Verification: Test Your Memory

Never trust a black box. Let's force the AI to show us what it found in the database. If this step fails, the agents will have nothing to say.

In [11]:
# 🧐 INSPECTION STATION

try:
    db = Chroma(
        client=persistent_client,
        collection_name="product_collection",
        embedding_function=embeddings
    )
    retriever = db.as_retriever(search_kwargs={"k": 2})

    # Let's ask a question relevant to products, not stores
    query = "is your product good?"
    results = retriever.invoke(query)

    print(f"\n🔎 Search Results for: '{query}'")
    for doc in results:
        print(f"--- Chunk ---\n{doc.page_content[:150]}...\n")

except Exception as e:
    print(f"⚠️ Error: {e}")


🔎 Search Results for: 'is your product good?'
--- Chunk ---
3. iPhone 16e - Starting Price: From S$949 - Key Specs: * 6.1" Super Retina XDR OLED * Compact design, Apple Intelligence - Link: https://www.apple.co...

--- Chunk ---
7. iPad (11", A16) - Starting Price: From S$499 - Key Specs: * A16 chip * All-screen Liquid Retina display * Supports Pencil & Magic Keyboard Folio - ...



---

## 6. The Brain: Building the Router

In a simple RAG system, every query goes to the same place. In an **Agentic System**, we first stop and ask: *"Who is best suited to answer this?"*

### The "Traffic Cop" Pattern
Our Router is a specialized prompt that classifies user intent. It does **not** answer the question; it outputs a **Decision**.

### Why not just one Agent?
* **Accuracy:** A "Shipping Agent" won't accidentally make up facts about "Product Specs."
* **Security:** You can prevent the "Refund Agent" from accessing "User Data."

In [12]:
# 🧠 THE ROUTER NODE
from langchain_core.messages import SystemMessage, HumanMessage

def router_node(state):
    query = state["query"]

    router_prompt = """
    You are a Router. Classify the user's query into one of these buckets:
    - 'product_agent': Buying, prices, specs, inventory, store locations.
    - 'policy_agent': Shipping, warranties, return rules, FAQ.
    - 'general_agent': Greetings, jokes, or off-topic queries.

    Output ONLY the exact key. If unsure, default to 'general_agent'.
    """

    try:
        response = llm.invoke([SystemMessage(content=router_prompt), HumanMessage(content=query)])
        decision = response.content.strip().lower().strip(".'\"")
    except:
        decision = "general_agent"

    if decision not in ["product_agent", "policy_agent", "general_agent"]:
        decision = "general_agent"

    return {"next_node": decision, "debug_log": f"🚦 Router Decision: {decision}"}


### The Router Lab
Let's test the "Brain" in isolation before connecting the body.

In [14]:
# 🧪 ROUTER LAB
test_queries = [
    "Do you have the iPhone 16?",
    "Can I return a broken item?",
    "Tell me a joke.",
    "Describe Singapore"
]

print(f"{'QUERY':<30} | {'DECISION':<15}")
print("-" * 50)
for q in test_queries:
    res = router_node({"query": q})
    print(f"{q:<30} | {res['next_node']:<15}")


QUERY                          | DECISION       
--------------------------------------------------
Do you have the iPhone 16?     | product_agent  
Can I return a broken item?    | policy_agent   
Tell me a joke.                | general_agent  
Describe Singapore             | general_agent  


---

## 7. The Specialists: Strict Grounding

Now we build the workers. We use a **Factory Pattern**: a single function `run_grounded_agent` that can act as any specialist depending on the inputs we give it.

**The "Strict Grounding" Rule:** If the retrieved documents are empty, the agent MUST admit ignorance. It is strictly forbidden from using outside knowledge for business facts.

In [15]:
# 🏭 AGENT FACTORY
def run_grounded_agent(state, agent_name, collection_name, persona):
    query = state["query"]

    # 1. Retrieve Data
    try:
        db = Chroma(client=persistent_client, collection_name=collection_name, embedding_function=embeddings)
        docs = db.similarity_search(query, k=3)
        context = "\n\n".join([d.page_content for d in docs])
    except:
        context = ""

    # 2. Strict Check
    if not context.strip():
        return {
            "response": "I could not find that information in our internal documents.",
            "debug_log": state.get('debug_log', '') + f"\n❌ {agent_name}: No context found."
        }

    # 3. Answer with Context
    system_prompt = f"""
    You are the {persona}.
    Answer the user query using ONLY the Context below.
    If the answer is not in the context, say you don't know.

    Context:
    {context}
    """

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=query)])

    return {
        "response": response.content,
        "debug_log": state.get('debug_log', '') + f"\n✅ {agent_name}: Used {len(docs)} docs."
    }

# Define the actual nodes
def product_agent_node(state):
    return run_grounded_agent(state, "Product Agent", "product_collection", "Sales Expert")

def policy_agent_node(state):
    return run_grounded_agent(state, "Policy Agent", "policy_collection", "Customer Support Specialist")

def general_agent_node(state):
    # General agent is allowed to chat freely
    res = llm.invoke([SystemMessage(content="You are a helpful assistant."), HumanMessage(content=state["query"])])
    return {
        "response": res.content,
        "debug_log": state.get('debug_log', '') + "\n💬 General Agent: Chatting freely."
    }


---

## 8. Orchestration: LangGraph

We have the Brain (Router) and the Body (Agents). Now we use **LangGraph** to wire them together into a state machine.

In [16]:
# 🕸️ BUILDING THE GRAPH
from typing import TypedDict, List
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    query: str
    response: str
    next_node: str
    debug_log: str

# Initialize Graph
workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("router", router_node)
workflow.add_node("product_agent", product_agent_node)
workflow.add_node("policy_agent", policy_agent_node)
workflow.add_node("general_agent", general_agent_node)

# Add Edges (The Wiring)
workflow.set_entry_point("router")
workflow.add_conditional_edges(
    "router",
    lambda state: state["next_node"] # The function that decides where to go
)
workflow.add_edge("product_agent", END)
workflow.add_edge("policy_agent", END)
workflow.add_edge("general_agent", END)

# Compile
app = workflow.compile()
print("✅ Agentic Graph Compiled.")


✅ Agentic Graph Compiled.


---

## 9. Testing the System

Let's run a full simulation in the console.

In [17]:
# 🕵️ SIMULATION
def test_agent(question):
    print(f"\n❓ Asking: {question}")
    result = app.invoke({"query": question})
    print(f"🤖 Answer: {result['response']}")
    print(f"🧠 Log: {result['debug_log']}")
    print("-" * 40)

test_agent("What are the latest products available?") # Should hit Product Agent
test_agent("What is the return deadline?")   # Should hit Policy Agent
test_agent("Write a haiku about AI.")        # Should hit General Agent



❓ Asking: What are the latest products available?
🤖 Answer: Here are the newest Apple products currently listed in the Singapore catalog (as of August 2025):

**iPhone**
1. **iPhone 16 Pro / Pro Max** – From S$1,599  
   *A18 Pro chip, 48 MP Fusion main camera, titanium frame, Apple Intelligence*  
   <https://www.apple.com/sg/iphone-16-pro/>

2. **iPhone 16 / 16 Plus** – From S$1,299  
   *A18 chip, dual‑camera (48 MP + 12 MP Ultra Wide), aluminium frame, Apple Intelligence*  
   <https://www.apple.com/sg/shop/buy-iphone/iphone-16>

3. **iPhone 16e** – From S$949  
   *6.1″ Super Retina XDR OLED, compact design, Apple Intelligence*  
   <https://www.apple.com/sg/shop/buy-iphone/iphone-16e>

4. **iPhone 15 / 15 Plus** – From ~S$1,099  
   *A16 Bionic chip, Dynamic Island, 48 MP main camera*  
   <https://www.apple.com/sg/shop/buy-iphone/iphone-15>

**iPad**
5. **iPad Pro (11″ & 13″, M4)** – From S$1,499  
   *M4 chip (up to 10‑core CPU/GPU), Ultra Retina XDR OLED, ProMotion, Pencil Pr

---

## 10. The User Interface (Gradio)

Finally, we wrap our system in a "Transparent UI." This allows users to chat normally, but also peek behind the curtain to see the **Thinking Process** (the `debug_log`).

In [ ]:
# 🎨 GRADIO UI
import gradio as gr

def chat_interface(message, history):
    try:
        result = app.invoke({"query": message})
        answer = result.get("response", "No response.")
        log = result.get("debug_log", "No log.")

        # Format output with a collapsible "Thinking" section
        display_text = f"{answer}\n\n<details><summary>🧠 <b>Thinking Process</b></summary>\n<pre>{log}</pre>\n</details>"
        return display_text
    except Exception as e:
        return f"Error: {str(e)}"

demo = gr.ChatInterface(
    fn=chat_interface,
    title="🤖 Agentic RAG Demo",
    description="Ask about products or policies. Expand the 'Thinking Process' to see the routing logic."
)

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f3175b0e33d73a07b2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

## 11. Debugging & Troubleshooting

If your agent fails, check these common issues:

| Symptom | Probable Cause | Fix |
| :--- | :--- | :--- |
| **"I could not find that..."** | Ingestion failed or file missing. | Check Section 5. Does the file exist in Drive? |
| **Router picks wrong agent** | Router prompt is ambiguous. | Edit `router_node` prompt definitions. |
| **Gradio Error** | State mismatch. | Ensure `AgentState` keys match what your nodes return. |

---

## 12. Extensions

How can you take this further?
1.  **Memory:** Add `history` to the `AgentState` so the bot remembers "it" refers to the iPhone you just mentioned.
2.  **Tools:** Give the Product Agent a calculator tool to sum up prices.
3.  **Guards:** Add a "Profanity Filter" node before the Router.

---

## 13. Summary

Today we built a system that **thinks before it speaks**.
* We replaced a single LLM with a **Graph of Agents**.
* We implemented **Semantic Routing** to direct traffic.
* We enforced **Strict Grounding** to prevent hallucinations.

This architecture is the foundation of modern Enterprise AI.